<a href="https://colab.research.google.com/github/yoh6ly/Yohaly/blob/main/intelligent_monitoring_of_renewable_and_electric_energy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Librerías necesarias ---
import csv
import datetime
import time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import serial
from sklearn.ensemble import IsolationForest

# --- Configuración del puerto serial ---
# Ajuste 'COM3' según el puerto real del Arduino/ESP
# Baud rate a 9600 para visualización estándar
arduino = serial.Serial('COM3', 9600, timeout=2)

# --- Lectura de sensores desde Arduino ---
def leer_sensores():
    """
    Lee datos enviados por Arduino en formato CSV: voltaje,corriente,potencia,humedad,temperatura
    """
    try:
        linea = arduino.readline().decode(errors="ignore").strip()
        if not linea:
            return None
        valores = linea.split(",")
        if len(valores) != 5:
            return None
        voltaje, corriente, potencia, humedad, temperatura = map(float, valores)
        return voltaje, corriente, potencia, humedad, temperatura
    except Exception as e:
        print("Error al leer sensores:", e)
        return None

# --- Registro de datos ---
def registrar_datos(nombre_archivo, v, c, p, h, t):
    fecha = datetime.datetime.now()
    with open(nombre_archivo, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([fecha, v, c, p, h, t])

# --- Cálculos electromagnéticos y de ondas ---
def calcular_electromagnetismo(v, c):
    """
    Calcula magnitudes básicas de electromagnetismo:
    - Potencia instantánea: P = V * I
    - Energía acumulada (aprox): E = P * Δt
    - Intensidad de campo eléctrico (simplificado): E = V/d (d=1m por defecto)
    """
    P = v * c
    E = P * 1  # Δt = 1s como base
    campo_electrico = v / 1.0
    return P, E, campo_electrico

def aplicar_fft(signal, threshold=0.1):
    """
    Aplica FFT para analizar la señal en frecuencia y filtrar ruido.
    """
    fft_vals = np.fft.fft(signal)
    fft_freqs = np.fft.fftfreq(len(signal))
    fft_vals[np.abs(fft_freqs) > threshold] = 0
    signal_filtrada = np.fft.ifft(fft_vals).real
    return signal_filtrada, fft_freqs, fft_vals

# --- Estadísticas y matrices ---
def calcular_estadisticas(datos):
    stats = datos.describe()
    print(" Estadísticas:\n", stats)
    return stats

def matriz_correlacion(datos):
    corr = datos.corr()
    print(" Matriz de correlación:\n", corr)
    plt.figure(figsize=(8,6))
    plt.imshow(corr, cmap="coolwarm", interpolation="none")
    plt.colorbar()
    plt.xticks(range(len(corr)), corr.columns, rotation=45)
    plt.yticks(range(len(corr)), corr.columns)
    plt.title("Matriz de correlación")
    plt.tight_layout()
    plt.show()
    return corr

# --- Confirmación de anomalías ---
def confirmar_anomalias(nombre_archivo, equipo="Ventilador"):
    datos = pd.read_csv(nombre_archivo, header=None,
                        names=["Fecha","Voltaje","Corriente","Potencia","Humedad","Temperatura"])

    # FFT sobre voltaje y corriente
    voltajes_filtrados, _, _ = aplicar_fft(datos["Voltaje"].values)
    corrientes_filtrados, _, _ = aplicar_fft(datos["Corriente"].values)

    datos["Voltaje_filtrado"] = voltajes_filtrados
    datos["Corriente_filtrado"] = corrientes_filtrados

    # Isolation Forest sobre señales filtradas
    X = datos[["Voltaje_filtrado","Corriente_filtrado","Potencia","Humedad","Temperatura"]]
    modelo = IsolationForest(contamination=0.1, random_state=42)
    etiquetas = modelo.fit_predict(X)
    datos["Estado"] = ["Anómalo" if e==-1 else "Normal" for e in etiquetas]

    datos.to_csv(nombre_archivo.replace(".csv","_confirmado.csv"), index=False)

    if etiquetas[-1] == -1:
        print(f" Anomalía confirmada en {equipo} (FFT + Isolation Forest)")
    else:
        print(f" Última lectura normal en {equipo}")

    return datos

# --- Ciclo de monitoreo ---
def ciclo_monitoreo(equipo="Ventilador"):
    archivo_diario = f"diario_{datetime.datetime.now().strftime('%Y%m%d')}.csv"
    tiempo_inicio_dia = time.time()

    while time.time() - tiempo_inicio_dia < 86400:  # 24h
        archivo_hora = f"reporte_{datetime.datetime.now().strftime('%Y%m%d_%H')}.csv"
        tiempo_inicio_hora = time.time()

        while time.time() - tiempo_inicio_hora < 3600:  # 1h
            lectura = leer_sensores()
            if lectura:
                v, c, p, h, t = lectura
                # Cálculos electromagnéticos
                P_calc, E_calc, campo_E = calcular_electromagnetismo(v, c)
                registrar_datos(archivo_hora, v, c, P_calc, h, t)
                registrar_datos(archivo_diario, v, c, P_calc, h, t)
            time.sleep(5)

        # Confirmación de anomalías y estadísticas
        confirmar_anomalias(archivo_hora, equipo)
        datos = pd.read_csv(archivo_hora, header=None,
                            names=["Fecha","Voltaje","Corriente","Potencia","Humedad","Temperatura"])
        calcular_estadisticas(datos)
        matriz_correlacion(datos)

    print(f" Exportación diaria completada para {equipo}")

# --- Ejecución del ciclo ---
ciclo_monitoreo("Ventilador")
ciclo_monitoreo("Motor")

